<a href="https://colab.research.google.com/github/SayanB58/HAAI__Cohort2_LLM_Tokenizers_Embeddings/blob/main/HAAI_Cohort2_Week4_AgenticAI_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This program is build with Flan-T5-XL LLM to be able to answer the final question using the output from the previous questions as in-context learning/few-shot learning.

Consider three related questions from a search session: Question 1, Question 2, Question 3
1. Answer to Question 1 needs to be generated.
2. Answer to Question 2 needs to be generated with the answer to Question 1 as one-shot example / context.
3. Answer to Question 3 needs to be generated with the answer to Question 2 as one-shot example / context.
4. Answer to Question 3 will be either YES or NO and nothing else.


* The program accepts three parameters provided as a command line input.
* The three inputs represent the questions.
* The output of the first two question is Generation based whereas the last question output is deterministic i.e. its either YES or NO.
* The output of the first two question is Generation based whereas the last question output is deterministic i.e. its either YES or NO.
* Output should be in upper-case: YES or NO
* There should be no additional output including any warning messages in the terminal.
* Remember that your output will be tested against test cases, therefore any deviation from the test cases will be considered incorrect during evaluation.

Syntax: python template.py `<string>` `<string>` `<string>`

The following example is given for your reference:

 Terminal Input: python template.py "Who is Rabindranath Tagore?" "Where was he born?" "Is it in America?"
Terminal Output: NO

 Terminal Input: python template.py "Who is Rabindranath Tagore?" "Where was he born?" "Is it in India?"
Terminal Output: YES

You are expected to create some examples of your own to test the correctness of your approach.

ALL THE BEST!!

ALERT: **No changes are allowed to import statements**

In [1]:
import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

##### You may comment this section to see verbose -- but you must un-comment this before final submission. ######
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
#################################################################################################################

In [2]:
from google.colab import userdata
#userdata.get('HF_TOKEN')

**Changes allowed from here**

In [3]:
def llm_function(model,tokenizer,questions):
    '''
    The steps are given for your reference:

    1. Generate answer for the first question.
    2. Generate answer for the second question use the answer for first question as context.
    3. Generate a deterministic output either 'YES' or 'NO' for the third question using the context from second question.
    5. Clean output and return.
    6. Output is case-sensative: YES or NO
    Note: The model (Flan-T5-XL) and tokenizer is already initialized. Do not modify that section.
    '''
    q1, q2, q3 = questions[0], questions[1], questions[2]

    prompt = f'''
    You are a careful question answering assistant.

    You will be given three related questions.

    Question 1: {q1}

    Question 2: {q2}

    Question 3: {q3}

    Perform the following steps internally.

    Step 1.
    Answer Question 1.

    Step 2.
    Use the answer from Step 1 as additional context to answer Question 2.

    Step 3.
    Use the answer from Step 2 as additional context to determine whether the answer to Question 3 is YES or NO.

    Important:
    - Perform all three steps internally.
    - Do not print the answer to Question 1.
    - Do not print the answer to Question 2.
    - Do not print your reasoning.
    - Do not print explanations.

    Output should be between YES and NO

    The output must contain exactly one uppercase word and nothing else.
    '''
    # print('Y ', tokenizer.encode("Y", return_tensors="pt", add_special_tokens=False))
    # print('N ', tokenizer.encode("N", return_tensors="pt", add_special_tokens=False))

    # Y = tokenizer.encode("Y", return_tensors="pt", add_special_tokens=False)[0].item()
    # N = tokenizer.encode("N", return_tensors="pt", add_special_tokens=False)[0].item()

    # print('Y ', Y)
    # print('N ', N)

    tokenized_prompt = tokenizer(prompt, return_tensors="pt").input_ids

    # print('tokenized_prompt ', tokenized_prompt)

    outputs = model.generate(tokenized_prompt, do_sample=False,  top_p=None, return_dict_in_generate=True, output_scores=True, max_new_tokens=3)

    # print('outputs ', outputs)

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    prediction = [predict_word.strip().upper() for predict_word in prediction]
    # print('prediction ',prediction)

    if "YES" in prediction:
        final_output = "YES"
    else:
        final_output = "NO"

    # Helper function to generate clean text from the model
    # def generate_response(prompt_text, max_new_tokens=64):
    #     inputs = tokenizer(prompt_text, return_tensors="pt")
    #     outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=2, early_stopping=True)
    #     return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # # Step 1: Generate answer to Question 1
    # prompt_1 = f"Answer the following question clearly:\nQuestion: {q1}\nAnswer:"
    # ans_1 = generate_response(prompt_1)

    # # Step 2: Generate answer to Question 2 using Question 1 and Answer 1 as one-shot context
    # prompt_2 = f"Context:\nQuestion: {q1}\nAnswer: {ans_1}\n\nTask:\nQuestion: {q2}\nAnswer:"
    # ans_2 = generate_response(prompt_2)

    # # Step 3: Force a deterministic YES/NO answer using Question 2 and Answer 2 as one-shot context
    # prompt_3 = (
    #     f"Context:\nQuestion: {q2}\nAnswer: {ans_2}\n\n"
    #     f"Task:\nBased on the context, answer the following question with exactly one word, either YES or NO.\n"
    #     f"Question: {q3}\nAnswer:"
    # )
    # ans_3 = generate_response(prompt_3, max_new_tokens=5)

    # # Clean up the output to make sure it only returns "YES" or "NO"
    # cleaned_output = ans_3.upper()
    # if "YES" in cleaned_output:
    #     final_output = "YES"
    # elif "NO" in cleaned_output:
    #     final_output = "NO"
    # else:
    #     final_output = "NO"  # Default fallback if structure is broken

    return final_output

ALERT: **No changes are allowed below this comment**

In [4]:
if __name__ == '__main__':

    # question_a = sys.argv[1].strip()
    # question_b = sys.argv[2].strip()
    # question_c = sys.argv[3].strip()

    question_a = "What is the Japanese yen?"
    question_b = "Which country uses it?"
    question_c = "Is that country in Africa?"

    questions = [question_a, question_b, question_c]
    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    # print(tokenizer.tokenize("YES"))
    # print(tokenizer.tokenize("NO"))
    out = llm_function(model,tokenizer,questions)
    print(out.strip())

    """ End to call """

KeyboardInterrupt: 